# 02 · Player (and ball candidate) detection

Default backend `yolox_onnx` (`football_eval/detection.py`): YOLOX-S trained on COCO, so it only knows *person* and
*sports ball*; roles (goalkeeper / referee) come later from the kits (04). Steps per frame:

1. **tile plan** — the fewest tiles of ≤ `detection.tile` px covering the region of interest, `tile_overlap` px overlap;
2. per tile: **letterbox** to 640×640 → network → 8400 anchors × (box, objectness, 80 class scores) → **decode**;
3. score = objectness × class prob, thresholds `conf_player` / `conf_ball`, **NMS** over all tiles;
4. **merge** people cut by an inner tile border; drop overlay regions and tiny boxes.

This notebook runs each step by hand, then the detector over the window, and shows how to score it against a
hand-annotated frame. To try another model, change `cfg.detection` (e.g. `backend: ultralytics`, a fine-tuned
checkpoint from 08) and re-run from here.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

## 1 · Build the detector
The runtime benchmark below is where the GPU is chosen (or not): every execution provider onnxruntime offers is timed on a dummy tile and the fastest is kept. If CUDA is installed but this prints `CUDAExecutionProvider unavailable`, see the fix branch (`python -m football_eval.accel --doctor`).

In [ ]:
from football_eval.detection import build_detector, tile_plan, merge_cut_boxes
detector = build_detector(cfg)
r = getattr(detector, "runner", None)
if r is not None:
    print(f"runtime: {r.name} | on GPU: {r.on_gpu} | {r.ms or float('nan'):.1f} ms per 640x640 tile (benchmark)")

## 2 · Tile plan: whole frame vs region of interest

In [ ]:
from football_eval.people import grass_top, grass_roi
f, t, img = nbt.probe_frame(ctx)
H_, W_ = img.shape[:2]
roi = grass_roi(grass_top(img), img.shape) if cfg.detection.pitch_roi else None
plans = {"whole frame": tile_plan(W_, H_, detector.tile, detector.overlap),
         "grass ROI": tile_plan(W_, H_, detector.tile, detector.overlap, roi)}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, (name, tiles) in zip(axes, plans.items()):
    vis = img.copy()
    for k, (x0, y0, x1, y1) in enumerate(tiles):
        cv2.rectangle(vis, (x0 + 3, y0 + 3), (x1 - 3, y1 - 3), nbt.COLORS["tile"], 5)
        cv2.putText(vis, f"tile {k}: {x1 - x0}x{y1 - y0}", (x0 + 20, y0 + 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 0, 255), 4)
    nbt.show(vis, f"{name}: {len(tiles)} tiles", ax=ax)
fig.tight_layout()

## 3 · One tile through the network, by hand
Letterbox (keep aspect, pad with grey 114) → blob (1×3×640×640, BGR, 0–255) → raw output 8400 × 85: `cx, cy, w, h` in grid units, objectness, 80 COCO class probabilities. Anchors come from 3 strides (8, 16, 32): 80² + 40² + 20² = 8400.

In [ ]:
x0, y0, x1, y1 = plans["grass ROI"][0]
tile = img[y0:y1, x0:x1]
S = detector.S
h, w = tile.shape[:2]; rr = min(S / h, S / w)
pad = np.full((S, S, 3), 114, np.uint8)
pad[:int(h * rr), :int(w * rr)] = cv2.resize(tile, (int(w * rr), int(h * rr)), interpolation=cv2.INTER_AREA)
blob = cv2.dnn.blobFromImage(pad, 1.0, (S, S), swapRB=False)
with nbt.Timer() as tm:
    raw = detector.runner(blob).copy()
print(f"tile {w}x{h} -> scale {rr:.3f} -> network input {blob.shape}, raw output {raw.shape} in {tm.s * 1000:.0f} ms")
o = raw.copy()
o[:, :2] = (o[:, :2] + detector.grids) * detector.strides          # grid cell + offset -> pixels (640 space)
o[:, 2:4] = np.exp(o[:, 2:4]) * detector.strides                  # log-size -> pixels
boxes = np.c_[o[:, 0] - o[:, 2] / 2, o[:, 1] - o[:, 3] / 2, o[:, 0] + o[:, 2] / 2, o[:, 1] + o[:, 3] / 2] / rr
s_person, s_ball = o[:, 4] * o[:, 5], o[:, 4] * o[:, 5 + detector.BALL_CLS]
print(f"anchors with person score > {cfg.detection.conf_player}: {(s_person > cfg.detection.conf_player).sum()}, "
      f"ball score > {cfg.detection.conf_ball}: {(s_ball > cfg.detection.conf_ball).sum()} (before NMS)")
# same result as the library's _infer
bp, sp, bb, sb = detector._infer(tile)
assert np.allclose(bp, boxes[s_person > detector.cp]) and np.allclose(sp, s_person[s_person > detector.cp])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
nbt.show(pad, "letterboxed network input", ax=axes[0])
axes[1].hist([s_person[s_person > 0.01], s_ball[s_ball > 0.01]], bins=40, log=True, label=["person", "sports ball"])
axes[1].axvline(cfg.detection.conf_player, color="C0", ls="--"); axes[1].axvline(cfg.detection.conf_ball, color="C1", ls="--")
axes[1].set_title("anchor scores (> 0.01) and the thresholds"); axes[1].legend();

## 4 · Threshold sweep and NMS on this tile
How many people / ball candidates survive each threshold, before and after non-maximum suppression (IoU 0.45). The plateau after NMS is the number of real people; where it starts to drop is the safe range for `conf_player`.

In [ ]:
def after_nms(b, s, thr):
    k = s > thr
    if not k.any(): return 0
    keep = cv2.dnn.NMSBoxes([[float(a), float(c), float(e - a), float(d - c)] for a, c, e, d in b[k]], s[k].tolist(), thr, 0.45)
    return len(np.array(keep).ravel())
thr = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sweep = pd.DataFrame({"person raw": [(s_person > q).sum() for q in thr], "person NMS": [after_nms(boxes, s_person, q) for q in thr],
                      "ball raw": [(s_ball > q).sum() for q in thr], "ball NMS": [after_nms(boxes, s_ball, q) for q in thr]}, index=thr)
sweep.index.name = "threshold"
sweep

## 5 · The full detector on the frame
All tiles + NMS + merging of people cut by a tile border. Box height vs foot row is a perspective sanity check: far players (top) must be smaller; outliers are usually crowd / staff or merged pairs.

In [ ]:
with nbt.Timer() as tm:
    det = detector(img, roi=roi)
people, balls = det.of("player", "goalkeeper", "referee"), det.of("ball")
print(f"{len(people)} people, {len(balls)} ball candidates, {len(detector.last_tiles)} tiles, {tm.s * 1000:.0f} ms")
vis = nbt.draw_boxes(img, people.xyxy, [f"{c:.2f}" for c in people.conf])
for (a, b, c, d), s in zip(balls.xyxy, balls.conf):
    cv2.circle(vis, (int((a + c) / 2), int((b + d) / 2)), 22, nbt.COLORS["ball"], 3)
nbt.show(vis, "people (white, score) and ball candidates (yellow)")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(people.xyxy[:, 3], people.xyxy[:, 3] - people.xyxy[:, 1], c=people.conf, cmap="viridis")
ax.set_xlabel("foot row v (px)"); ax.set_ylabel("box height (px)"); ax.set_title("perspective check");

## 6 · Over the whole window
The same call on every analysed frame (ROI from the grass mask; the integrated loop uses the previous frame's pitch outline once calibrated). Timing per frame tells you the cost of a full match.

In [ ]:
rows, ms = [], []
for fi, ti, im in nbt.frames_in_window(ctx):
    r_ = grass_roi(grass_top(im), im.shape) if cfg.detection.pitch_roi else None
    t0 = time.perf_counter(); d = detector(im, roi=r_); dt = (time.perf_counter() - t0) * 1000
    ms.append(dt)
    for (a, b, c, e), s, k in zip(d.xyxy, d.conf, d.cls):
        rows.append((fi, ti, k, float(s), a, b, c, e, len(detector.last_tiles), dt))
dets = pd.DataFrame(rows, columns=["frame", "t", "cls", "conf", "x0", "y0", "x1", "y1", "n_tiles", "ms"])
per = dets[dets.cls != "ball"].groupby("frame").size()
print(f"{len(ms)} frames: {np.mean(ms):.0f} ms/frame ({1000 / np.mean(ms):.1f} fps), people/frame median {per.median():.0f} "
      f"[{per.min()}-{per.max()}], ball candidate in {dets[dets.cls == 'ball'].frame.nunique() / len(ms):.0%} of the frames")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
per.reset_index(drop=True).plot(ax=axes[0], title="people per frame")
dets[dets.cls != "ball"].conf.hist(bins=30, ax=axes[1]); axes[1].set_title("people scores")
pd.Series(ms).plot(ax=axes[2], title="ms per frame");

## 7 · Scoring against a hand-annotated frame
Format of `data/annotations/<name>.json` (see `football_eval/evaluation.py`): `{"image", "frame", "boxes": [{"xyxy", "cls", "team"}], "ball": [u, v]}`.
The cell writes a **template from the detections** for the probe frame if none exists (plus the image), so you only
correct it (delete false boxes, add missed players, fix the ball) instead of drawing from scratch. Re-run the cell
afterwards to get precision / recall at IoU 0.5 and the ball distance.

In [ ]:
from football_eval.evaluation import detection_pr
ann_dir = ROOT / "data" / "annotations"; ann_dir.mkdir(parents=True, exist_ok=True)
ann_path = ann_dir / f"{P.run_name}_f{f}.json"
if not ann_path.exists():
    cv2.imwrite(str(ann_path.with_suffix(".png")), img)
    tpl = {"image": ann_path.with_suffix(".png").name, "video": Path(ctx.video).name, "frame": int(f), "t": float(t),
           "status": "template from detections - CORRECT ME, then set status to 'checked'",
           "boxes": [{"xyxy": [round(float(v), 1) for v in b], "cls": c, "team": -1} for b, c in zip(people.xyxy, people.cls)],
           "ball": [round(float((balls.xyxy[0, 0] + balls.xyxy[0, 2]) / 2), 1), round(float((balls.xyxy[0, 1] + balls.xyxy[0, 3]) / 2), 1)] if len(balls) else None}
    ann_path.write_text(json.dumps(tpl, indent=1), encoding="utf-8")
    print("template written:", ann_path.relative_to(ROOT), "-> correct it (any JSON editor, or LabelMe / CVAT import) and re-run")
ann = json.loads(ann_path.read_text(encoding="utf-8"))
gt = np.array([b["xyxy"] for b in ann["boxes"]], float).reshape(-1, 4)
res = {thr_: detection_pr(people.xyxy, gt, thr_) for thr_ in (0.3, 0.5, 0.7)}
print("annotation status:", ann.get("status", "checked"))
out = pd.DataFrame({k: {m: v[m] for m in ("tp", "fp", "fn", "precision", "recall")} for k, v in res.items()}).T
out.index.name = "IoU threshold"
if ann.get("ball") and len(balls):
    bc = np.c_[(balls.xyxy[:, 0] + balls.xyxy[:, 2]) / 2, (balls.xyxy[:, 1] + balls.xyxy[:, 3]) / 2]
    print(f"ball: closest candidate {np.min(np.hypot(*(bc - np.array(ann['ball'])).T)):.1f} px from the annotated ball")
out

In [ ]:
nbt.save_stage(ctx, "detections", dets);